# 01 — Eksplorasi Data Awal (Data Exploration)
**NapasLoka: Sistem Prediksi Polutan Udara (CO & CO₂) Wilayah Jabodetabek**

Notebook ini bertujuan untuk melakukan inspeksi awal terhadap kualitas, kelengkapan temporal, dan karakteristik distribusi data polutan serta parameter meteorologi dari 15 stasiun pemantau di Jabodetabek:
1. Memuat dataset mentah CO dan CO₂.
2. Memeriksa standardisasi zona waktu (WIB / UTC+7).
3. Mengevaluasi integritas data (missing values, duplikasi, dan rentang tanggal).
4. Analisis statistik deskriptif dan visualisasi sebaran polutan.


---
### 1. Setup Lingkungan & Deteksi Platform (Colab vs Lokal)


In [ ]:
import os
import sys
from pathlib import Path

# Deteksi lingkungan eksekusi
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("--> Lingkungan: Google Colab.")
    if not os.path.exists('napasloka'):
        !git clone https://github.com/Nikanzz/napasloka.git
    %cd napasloka
    !pip install -q xgboost lightgbm shap pyyaml joblib matplotlib plotly scikit-learn
    ROOT_DIR = Path("/content/napasloka")
else:
    ROOT_DIR = Path.cwd()
    if not (ROOT_DIR / "data").exists() and (ROOT_DIR.parent / "data").exists():
        ROOT_DIR = ROOT_DIR.parent
    print(f"--> Lingkungan: Lokal. Root direktori: {ROOT_DIR}")

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print("✓ Modul src berhasil disiapkan.")


---
### 2. Pemuatan Modul & Pembacaan Dataset


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import load_yaml_config

# Memuat konfigurasi stasiun
locations_cfg = load_yaml_config("locations.yaml").get("locations", [])
print(f"Total stasiun terkonfigurasi: {len(locations_cfg)} stasiun")

# Membaca dataset terproses
df_co = pd.read_csv(ROOT_DIR / "data" / "processed" / "jabodetabek_co_dataset.csv")
df_co2 = pd.read_csv(ROOT_DIR / "data" / "processed" / "jabodetabek_co2_dataset.csv")

# Standardisasi datetime
df_co["datetime"] = pd.to_datetime(df_co["datetime"], utc=True).dt.tz_convert("Asia/Jakarta")
df_co2["datetime"] = pd.to_datetime(df_co2["datetime"], utc=True).dt.tz_convert("Asia/Jakarta")

print(f"Dataset CO  : {len(df_co):,} baris | Rentang: {df_co['datetime'].min()} s.d. {df_co['datetime'].max()}")
print(f"Dataset CO₂ : {len(df_co2):,} baris | Rentang: {df_co2['datetime'].min()} s.d. {df_co2['datetime'].max()}")


---
### 3. Pemeriksaan Missing Values & Kelengkapan Data


In [ ]:
print("=== MISSING VALUES DATASET CO ===")
print(df_co.isnull().sum())

print("
=== MISSING VALUES DATASET CO₂ ===")
print(df_co2.isnull().sum())

# Pemeriksaan jumlah baris per stasiun
stasiun_counts = df_co.groupby("location").size()
print(f"
Jumlah observasi per stasiun (CO): {stasiun_counts.unique().tolist()} baris per lokasi")


---
### 4. Statistik Deskriptif Konsentrasi Polutan & Cuaca


In [ ]:
cols_co = ["carbon_monoxide", "temperature_2m", "relative_humidity_2m", "rain", "wind_speed_100m"]
display(df_co[cols_co].describe().round(2))

cols_co2 = ["carbon_dioxide", "temperature_2m", "relative_humidity_2m", "rain", "wind_speed_100m"]
display(df_co2[cols_co2].describe().round(2))


---
### 5. Visualisasi Distribusi & Korelasi Polutan


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribusi CO
axes[0].hist(df_co["carbon_monoxide"].dropna(), bins=50, color="#ea580c", alpha=0.75, edgecolor="black")
axes[0].set_title("Distribusi Konsentrasi CO (µg/m³)", fontsize=12)
axes[0].set_xlabel("Konsentrasi CO")
axes[0].set_ylabel("Frekuensi")
axes[0].grid(True, linestyle=":", alpha=0.6)

# Distribusi CO2
axes[1].hist(df_co2["carbon_dioxide"].dropna(), bins=50, color="#0284c7", alpha=0.75, edgecolor="black")
axes[1].set_title("Distribusi Konsentrasi CO₂ (ppm)", fontsize=12)
axes[1].set_xlabel("Konsentrasi CO₂")
axes[1].set_ylabel("Frekuensi")
axes[1].grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

# Heatmap Korelasi Meteorologi terhadap CO2
plt.figure(figsize=(8, 6))
corr = df_co2[cols_co2].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", cbar=True)
plt.title("Matriks Korelasi CO₂ dan Variabel Meteorologi", fontsize=12, pad=12)
plt.tight_layout()
plt.show()
